> **Not part of the final analysis.** This notebook sums *total* oil palm area (industrial and smallholder) for every Indonesian village on the 2021 BPS boundary layer (`PDesFin`). The published CVI instead uses oil palm on peat and village area from the same **Ina-Geoportal** village polygons (see `../00d_prepare_analysis_input.py`), so that Exposure has one consistent boundary set. The notebook is kept as a reusable national oil-palm-by-village workflow.

# 00 · Oil palm area by village (Google Earth Engine, Python API in Colab)

Part of the repository for *Composite Vulnerability Index of Villages with Oil Palm on Peatland in Indonesia in 2021* (JIMDP).

**What it does.** For every village in Indonesia, it sums the 2021 oil palm area from Descals et al. (2024) by plantation type and exports one CSV per province to Google Drive. It then merges these into one national table.

| | |
|---|---|
| **Oil palm layer** | `projects/sat-io/open-datasets/global-oil-palm/GlobalOilPalm_extent_2021` (Descals et al., 2024; 10 m, Sentinel-1; 1 = industrial, 2 = smallholder) |
| **Village boundaries** | 2021 BPS village layer uploaded as an Earth Engine asset (`PDesFin`) |
| **Outputs** | `VillageArea_Ha`, `Total_Palm_Ha`, `Ind_Palm_Ha`, `Small_Palm_Ha` per village (`KDEPUM`) |
| **Processing scale** | 30 m (`SCALE`), with `tileScale = 16`, to stay within Earth Engine memory limits for ~84,000 villages |

**Why Colab rather than the Code Editor.** At national scale a single `reduceRegions` call exceeds the Code Editor's limits. The job is therefore split into one export task per province (and per regency if a province has more than `MAX_VILLAGES_PER_BATCH` villages), and the tasks are run from Python.

**Oil palm on peatland** (`peat_tp_ha`, `peat_ip_ha`, `peat_sp_ha`, used for Exposure) was computed in a separate step that overlays the oil palm layer on the BIG peatland map; see `00b_…` and the README.

**Notes on interpreting the output**
* Areas are summed at 30 m, so the 10 m oil palm map is resampled. Village totals can therefore differ slightly from a 10 m calculation, by up to a few hectares in small villages.
* `KDEPUM` is not unique in the 2021 BPS layer: about 4,000 codes are shared by different villages. Duplicate codes are **different villages, not duplicated exports**, so all rows are kept here. The analysis (script `01`) joins the Village Development Index (IDM) by village code and drops the 22 codes that remain duplicated in the study sample.

## 1 · Set-up

In [ ]:
# Earth Engine Python API (usually pre-installed in Colab; upgrade to be safe)
!pip install -q earthengine-api --upgrade

import ee, time, glob, os
from collections import Counter

PROJECT_ID = 'sinta1'                       # <-- your Google Cloud project with Earth Engine enabled
ee.Authenticate()                           # opens a browser sign-in the first time
ee.Initialize(project=PROJECT_ID)

In [ ]:
# ----------------------------------------------------------------------------- parameters
VILLAGE_ASSET = f'projects/{PROJECT_ID}/assets/PDesFin'          # 2021 BPS village boundaries
OIL_PALM = 'projects/sat-io/open-datasets/global-oil-palm/GlobalOilPalm_extent_2021'
SCALE = 30                     # m; processing scale for the zonal sums
TILE_SCALE = 16                # higher = less memory per tile, slower
MAX_VILLAGES_PER_BATCH = 9000  # provinces above this are split by regency
DRIVE_FOLDER = 'CVI_oilpalm_exports'   # all export CSVs go to this Drive folder
PREFIX = 'OilPalm_Area_'

## 2 · Village areas and oil palm area images

Village area is the geodesic area of each polygon. Oil palm area uses three masked pixel-area bands, so a single `reduceRegions(sum)` call returns all three columns. No grouped reducer is needed; the earlier grouped-reducer version failed with `List.filter` errors.

In [ ]:
villages = ee.FeatureCollection(VILLAGE_ASSET)
villages = villages.map(lambda f: f.set('VillageArea_Ha', f.geometry().area(1).divide(10000)))

palm = ee.ImageCollection(OIL_PALM).mosaic().clip(villages.geometry().bounds())
pix_ha = ee.Image.pixelArea().divide(10000)

palm_area = (pix_ha.updateMask(palm.eq(1)).rename('Ind_Palm_Ha')
             .addBands(pix_ha.updateMask(palm.eq(2)).rename('Small_Palm_Ha'))
             .addBands(pix_ha.updateMask(palm.gt(0)).rename('Total_Palm_Ha')))

## 3 · Build and start one export task per province

In [ ]:
COLS = ['KDEPUM', 'nmdesa', 'nmkab', 'nmprov', 'VillageArea_Ha', 'Total_Palm_Ha', 'Ind_Palm_Ha', 'Small_Palm_Ha']

def export_task(subset, tag):
    stats = palm_area.reduceRegions(collection=subset, reducer=ee.Reducer.sum(),
                                    scale=SCALE, tileScale=TILE_SCALE)
    # villages without oil palm return no value; set them to 0
    def fill(f):
        return f.set({c: ee.Algorithms.If(f.get(c), f.get(c), 0)
                      for c in ['Ind_Palm_Ha', 'Small_Palm_Ha', 'Total_Palm_Ha']})
    return ee.batch.Export.table.toDrive(collection=stats.map(fill), description=PREFIX + tag,
                                         folder=DRIVE_FOLDER, fileFormat='CSV', selectors=COLS)

tasks = []
for prov in villages.aggregate_array('kdprov').distinct().sort().getInfo():
    sub = villages.filter(ee.Filter.eq('kdprov', prov))
    n = sub.size().getInfo()
    if n <= MAX_VILLAGES_PER_BATCH:
        tasks.append(export_task(sub, f'Prov{prov}'))
    else:
        for kab in sub.aggregate_array('idkab').distinct().getInfo():
            tasks.append(export_task(sub.filter(ee.Filter.eq('idkab', kab)), f'Prov{prov}_Kab{kab}'))
    print(f'Province {prov}: {n} villages')

for i, t in enumerate(tasks, 1):
    t.start(); time.sleep(1)
    print(f'Started {i}/{len(tasks)}: {t.config["description"]}')

## 4 · Monitor the tasks

Re-run this cell until every task is `COMPLETED`. Earth Engine runs only a few tasks at a time per account, so the national run can take several hours.

In [ ]:
mine = [t for t in ee.batch.Task.list() if t.status()['description'].startswith(PREFIX)]
latest = {}
for t in mine:                       # keep the most recent task per description
    s = t.status(); d = s['description']
    if d not in latest or s['creation_timestamp_ms'] > latest[d]['creation_timestamp_ms']:
        latest[d] = s
print(Counter(s['state'] for s in latest.values()))
for d, s in sorted(latest.items()):
    if s['state'] == 'FAILED':
        print(d, '->', s.get('error_message'))

## 5 · Merge the province files into one national table

In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')

files = sorted(glob.glob(f'/content/drive/MyDrive/{DRIVE_FOLDER}/{PREFIX}Prov*.csv'))
print(f'{len(files)} files')
nat = pd.concat([pd.read_csv(f).assign(source_file=os.path.basename(f)) for f in files], ignore_index=True)

# checks
print('villages (rows):', len(nat))
print('rows in more than one file (should be 0):', nat.duplicated(subset=['KDEPUM', 'nmdesa', 'nmkab']).sum())
print('KDEPUM codes shared by different villages:', nat['KDEPUM'].duplicated().sum(), '(expected; see notes)')
print('max |Ind + Small - Total| (ha):', round((nat.Ind_Palm_Ha + nat.Small_Palm_Ha - nat.Total_Palm_Ha).abs().max(), 4))
print(f"national oil palm: {nat.Total_Palm_Ha.sum():,.0f} ha "
      f"(industrial {nat.Ind_Palm_Ha.sum():,.0f}; smallholder {nat.Small_Palm_Ha.sum():,.0f})")

nat.drop(columns='source_file').to_csv(f'/content/drive/MyDrive/{DRIVE_FOLDER}/OilPalm_Area_National.csv', index=False)
print('saved OilPalm_Area_National.csv')

## Results of the original run (2026)

* 34 province tasks completed and were merged into 84,093 village rows.
* National oil palm area: 13,802,318 ha (industrial 9,818,612 ha; smallholder 3,983,706 ha).
* (Ind + Small) − Total = 0 for every village.
* 4,087 `KDEPUM` codes are shared by different villages in the boundary layer.

The table `OilPalm_Area_National.csv` was joined to the IDM 2021 records and to the oil-palm-on-peat areas to form `final_merged_dataset.csv`, the input to scripts `01`–`04`.